In [0]:
import json
import pandas as pd
from datetime import datetime

In [0]:
# Load signals
signals_df = spark.table("p2_signals").toPandas()
signals_df['Date'] = pd.to_datetime(signals_df['Date'])
aapl = signals_df[signals_df['Ticker'] == 'AAPL'].sort_values('Date')

latest_date = aapl['Date'].max()
today_sig   = aapl[aapl['Date'] == latest_date].iloc[0]

# Load OHLCV
raw_df = spark.table("p2_market_raw_universe").toPandas()
raw_df['Date'] = pd.to_datetime(raw_df['Date'])
raw_df = raw_df.sort_values('Date')

today_row = raw_df[raw_df['Date'] == latest_date]
prev_row  = raw_df[raw_df['Date'] < latest_date].iloc[-1]

if len(today_row) > 0:
    today_row  = today_row.iloc[0]
    open_price = round(float(today_row['Open_AAPL']),  2)
    close_price= round(float(today_row['Close_AAPL']), 2)
    change_pct = round((close_price - float(prev_row['Close_AAPL'])) / float(prev_row['Close_AAPL']) * 100, 2)
else:
    open_price = close_price = change_pct = None

# Yesterday's prediction vs actual
prev_dates   = aapl[aapl['Date'] < latest_date]
yesterday_result = None
if len(prev_dates) > 0:
    yest_sig  = prev_dates.iloc[-1]
    yest_pred = yest_sig['signal_side']
    if change_pct is not None:
        correct = (yest_pred == 'LONG' and change_pct > 0) or \
                  (yest_pred == 'SHORT' and change_pct < 0) or \
                  (yest_pred == 'FLAT')
        yesterday_result = {
            'prediction':       yest_pred if yest_pred != 'FLAT' else 'NO_TRADE',
            'actual_change_pct':change_pct,
            'correct':          bool(correct)
        }

# Confidence label
prob = float(today_sig['predicted_probability'])
conf = 'High' if prob >= 0.70 else ('Medium' if prob >= 0.60 else 'Low')
side = today_sig['signal_side']

output = {
    'last_updated': latest_date.strftime('%Y-%m-%d'),
    'signals': [{
        'ticker':      'AAPL',
        'signal':      side if side != 'FLAT' else 'NO_TRADE',
        'probability': round(prob, 4),
        'confidence':  conf,
        'direction':   'UP' if side == 'LONG' else ('DOWN' if side == 'SHORT' else 'FLAT')
    }],
    'prices': {
        'open':       open_price,
        'close':      close_price,
        'change_pct': change_pct
    },
    'yesterday_result': yesterday_result,
    'backtest': {
        'AAPL': {'hit_rate': 0.58, 'sharpe': 1.23, 'cumulative_return': 34.2, 'trades': 47}
    },
    'feature_importance': [
        {'feature': '5d Return',       'importance': 18},
        {'feature': 'VIX Level',       'importance': 15},
        {'feature': 'News Sentiment',  'importance': 13},
        {'feature': 'RSI 14',          'importance': 11},
        {'feature': 'Volume Ratio',    'importance':  9},
        {'feature': 'MA Cross 20/50',  'importance':  8},
        {'feature': 'SPY Correlation', 'importance':  7},
        {'feature': 'Volatility 20d',  'importance':  6},
    ],
    'cumulative_returns': {
        'dates': ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
        'AAPL': [0, 3, 7, 11, 14, 18, 21, 25, 28, 30, 33, 34],
        'TSLA': [0, 2, 4,  7,  9, 11, 13, 15, 17, 18, 20, 21],
        'MSFT': [0, 4, 8, 13, 17, 21, 25, 29, 33, 36, 39, 41],
    }
}

import os
os.makedirs("/Workspace/Users/ariamostajeran99@gmail.com/portfolio", exist_ok=True)
with open("/Workspace/Users/ariamostajeran99@gmail.com/portfolio/latest_signals.json", "w") as f:
    f.write(json.dumps(output, indent=2))
print("✓ Written to Workspace")
print(f"✓ Exported — Signal: {side} ({prob:.1%}), AAPL: ${open_price} → ${close_price} ({change_pct:+.2f}%)")
if yesterday_result:
    print(f"  Yesterday: {yesterday_result['prediction']} → {yesterday_result['actual_change_pct']:+.2f}% ({'✓' if yesterday_result['correct'] else '✗'})")
